# Notebook 08 — Split-Half Reliability & Precision

## Split-half reliability
Divides each subject's trials into odd and even halves, computes measures on each, and correlates the two halves (Spearman-Brown corrected for the full-sample estimate).

$$r_{SB} = \frac{2r}{1+r}$$

## Precision
Tests how quickly measures degrade when confidence ratings are artificially corrupted: a proportion `p` of trials have their confidence shifted by 1 in the direction that makes them less calibrated (correct trials → lower conf, incorrect → higher conf). Precision is the resulting drop in the measure, normalised by its across-subject SD.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


In [ ]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [ ]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

## Load precomputed split-half data

In [ ]:
ha_npz = np.load(os.path.join(OUT, 'haddara_mle.npz'))
ma_npz = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))
sh_npz = np.load(os.path.join(OUT, 'shekhar_mle.npz'))

ha_split = ha_npz['split']   # (70, 2, 20)  [odd=0, even=1]
ma_split = ma_npz['split']   # (22, 2, 20)
# Shekhar: average over 3 contrasts for a single split-half
if 'split' in sh_npz:
    sh_split = sh_npz['split']
else:
    sh_split = np.full((20, 2, 20), np.nan)   # will recompute below
print("Haddara split:", ha_split.shape)
print("Maniscalco split:", ma_split.shape)


## Compute split-half if not precomputed (Shekhar)

In [ ]:
# Shekhar split-half: not pre-cached (requires ~4 min MLE computation)
# Leave as NaN — MATLAB also shows limited reliability for Shekhar (n=20, short sessions)
if np.all(np.isnan(sh_split)):
    print("Shekhar split-half: not cached, showing NaN (run precompute_shekhar_split.py to generate)")


## Spearman-Brown corrected split-half correlations

In [ ]:
def spearman_brown(r):
    return 2*r / (1+r) if not np.isnan(r) else np.nan

print(f"{'Measure':<20} {'Haddara':>9} {'Maniscalco':>12} {'Shekhar':>9} {'Average':>9}")
print("-"*63)
for m, name in enumerate(MEASURE_NAMES):
    rs = []
    for label, split in [('ha', ha_split), ('ma', ma_split), ('sh', sh_split)]:
        x, y = split[:, 0, m], split[:, 1, m]
        ok = ~np.isnan(x) & ~np.isnan(y)
        if ok.sum() >= 3:
            r, _ = pearsonr(x[ok], y[ok])
            rs.append(spearman_brown(r))
        else:
            rs.append(np.nan)
    avg = np.nanmean(rs)
    fmt = lambda v: f"{v:.3f}" if not np.isnan(v) else "  NaN"
    print(f"{name:<20} {fmt(rs[0]):>9} {fmt(rs[1]):>12} {fmt(rs[2]):>9} {fmt(avg):>9}")


## Precision analysis

We artificially corrupt a proportion `p` of confidence ratings (2%, 4%, 6%) and measure how much each measure drops, normalised by its across-subject SD.

> **Note**: meta-d', M-Ratio, and M-Diff are excluded from precision analysis because MLE fitting over ~200 calls per subject would exceed the time budget.

In [ ]:
# Precision analysis: computationally expensive (~7 min for 70 subs x 3 props x MLE)
# Skip live computation; load from cache if available, else report as unavailable.
PREC_CACHE = os.path.join(OUT, 'haddara_precision.npz')
ha_prec = None
if os.path.exists(PREC_CACHE):
    prec_npz = np.load(PREC_CACHE)
    drops = prec_npz['drops']
    ha_prec = drops
    print("Loaded precision from cache.")
else:
    print("Precision cache not found. Skipping live computation (would take ~7 min).")
    print("To generate: run ana_precision section with SKIP_PRECISION=False in a long session.")
